In [ ]:
import cartopy.crs as ccrs

PROJECTION = ccrs.LambertConformal(
    central_longitude=25.0,
    central_latitude=56.7,
    standard_parallels=[56.7, 56.7],
    globe=ccrs.Globe(
        semimajor_axis=6367470.0,
        semiminor_axis=6367470.0,
    ),
)

BORDER_WIDTH = 400000 # in m
ZOOM = 1  # Unused


In [ ]:
def set_map_extent(ax, zoom_factor=None, boundary_data=None):
    """Set the map extent based on zoom factor and boundary data."""
    # Set new extent
    ax.set_extent(
        [
            boundary_data.x.min().values - BORDER_WIDTH,
            boundary_data.x.max().values + BORDER_WIDTH,
            boundary_data.y.min().values - BORDER_WIDTH,
            boundary_data.y.max().values + BORDER_WIDTH,
        ],
        crs=PROJECTION,
    )

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import pandas as pd
import matplotlib.patches as mpatches

def plot_field(ax, data, vmin, vmax):
    return ax.pcolormesh(
        data.longitude if hasattr(data, "longitude") else data.lon,
        data.latitude if hasattr(data, "latitude") else data.lat,
        data.values,
        transform=ccrs.PlateCarree(),
        vmin=vmin,
        vmax=vmax,
        cmap="viridis",
        shading="auto",
        rasterized=True,
    )

def plot_interior_vs_boundary(
    ds_interior: xr.Dataset,
    ds_boundary: xr.Dataset,
    interior_var: str,
    boundary_var: str,
    plot_time=None,
    zoom_factor=None,
    projection=ccrs.PlateCarree(),
    dpi=100,
):
    """
    Plot a single variable from two Zarr datasets (interior and boundary) for a selected time.
    """
    # Select time
    if plot_time is None:
        time_selected = ds_interior.time.values[0]
    else:
        time_selected = plot_time

    arr_int = ds_interior["state"].sel(state_feature=interior_var).sel(time=time_selected)
    arr_bnd  = ds_boundary["forcing"].sel(forcing_feature=boundary_var).sel(time=time_selected)

    print(f"arr_int: {arr_int.values.shape}")
    print(f"arr_bnd: {arr_bnd.values.shape}")

    # Compute global vmin/vmax for color scale
    vmin = np.nanmin([arr_int.values.min(), arr_bnd.values.min()])
    vmax = np.nanmax([arr_int.values.max(), arr_bnd.values.max()])

    fig = plt.figure(figsize=(14, 6), dpi=dpi)

    # First subplot with PlateCarree
    ax0 = fig.add_subplot(1, 2, 1, projection=ccrs.PlateCarree())
    # Second subplot with LambertConformal
    ax1 = fig.add_subplot(1, 2, 2, projection=ccrs.LambertConformal(
        central_longitude=25.0,
        central_latitude=56.7,
        standard_parallels=[56.7, 56.7],
        globe=ccrs.Globe(
            semimajor_axis=6367470.0,
            semiminor_axis=6367470.0,
        ),
    ))

    axes = [ax0, ax1]

    # Unstack to 2D using xarray if possible
    if "grid_index" in arr_int.dims and "lat" in ds_interior.coords and "lon" in ds_interior.coords:
        arr_int_2d = arr_int.set_index(grid_index=["y", "x"]).unstack("grid_index")
        arr_bnd_2d = arr_bnd.set_index(grid_index=["latitude", "longitude"]).unstack("grid_index")
        longitude_new = np.where(
        arr_bnd_2d["longitude"] > 180,
        arr_bnd_2d["longitude"] - 360,
        arr_bnd_2d["longitude"],
        )
        arr_bnd_2d = arr_bnd_2d.assign_coords(longitude=longitude_new).sortby([
            "longitude",
            "latitude",
        ])

        # For boundary
        lon_mesh_bnd, lat_mesh_bnd = np.meshgrid(arr_bnd_2d.longitude, arr_bnd_2d.latitude)
    else:
        raise ValueError("Data is not in grid_index/lat/lon format.")

    # Now arr_int_2d and arr_bnd_2d are 2D, and lon_mesh, lat_mesh are 2D
    print(f"arr_int_2d: {arr_int_2d.values.shape}")
    print(f"arr_bnd_2d: {arr_bnd_2d.values.shape}")



    # Plot interior
    im0 = plot_field(axes[0], arr_int_2d, vmin, vmax)
    plot_field(axes[0], arr_bnd_2d, vmin, vmax)

    # if zoom_factor is not None:
    #     set_map_extent(axes[0], zoom_factor, arr_int)
    
    axes[0].set_title(f"ERA5 PROJECTION PlateCarree")
    axes[0].coastlines()
 
    # # Plot boundary
    # axes[0].contourf(
    #     lon_mesh_bnd, lat_mesh_bnd, arr_bnd_2d.values,
    #         transform=ccrs.PlateCarree(),
    #         vmin=vmin,
    #         vmax=vmax,
    #         cmap="viridis",
    #         levels=200,
    #     )   
    if zoom_factor is not None:
        set_map_extent(axes[0], zoom_factor, arr_int_2d)

    # # Plot interior
    im0 = plot_field(axes[1], arr_int_2d, vmin, vmax)
    # plot_field(axes[1], arr_bnd_2d, vmin, vmax)

    # # Plot boundary
    # axes[1].contourf(
    #     lon_mesh_bnd, lat_mesh_bnd, arr_bnd_2d.values,
    #         transform=ccrs.PlateCarree(),  
    #         vmin=vmin,
    #         vmax=vmax,
    #         cmap="viridis",
    #         levels=200,
    #     )  

    if zoom_factor is not None:
        set_map_extent(axes[1], zoom_factor, arr_int_2d)

    axes[1].set_title(f"DANRA PROEJCTION LambertConformal")
    axes[1].coastlines()

    for ax in axes:
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.x_inline = False  # <-- Add this
        gl.y_inline = False  # <-- Add this
        ax.set_xlabel("Longitude")
        ax.set_ylabel("Latitude")

    # Get the bounding box of the interior grid in lat/lon
    # lat_min, lat_max = arr_int_2d.lat.values.min(), arr_int_2d.lat.values.max()
    # lon_min, lon_max = arr_int_2d.lon.values.min(), arr_int_2d.lon.values.max()

    # for ax in axes:
    #     # Create a rectangle in PlateCarree coordinates
    #     rect = mpatches.Rectangle(
    #         (lon_min, lat_min),
    #         lon_max - lon_min,
    #         lat_max - lat_min,
    #         linewidth=2,
    #         edgecolor='red',
    #         facecolor='none',
    #         transform=ccrs.PlateCarree(),
    #         zorder=10,
    #         label='Interior bounds'
    #     )
    #     ax.add_patch(rect)

    # Add colorbar
    cbar_ax = fig.add_axes([0.2, 0.02, 0.6, 0.02])
    fig.colorbar(im0, cax=cbar_ax, orientation="horizontal", label=interior_var)

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.show()

In [ ]:
ds_interior = xr.open_zarr("/proj/berzelius-2022-164/users/x_erila/real-prob-lam/neural-lam-dev-building-ml-lams/data/mdp_processed_data/danra_model1_config.zarr")
ds_boundary = xr.open_zarr("/proj/berzelius-2022-164/users/x_erila/real-prob-lam/neural-lam-dev-building-ml-lams/data/mdp_processed_data/overlapping_era_400km_model1_config.zarr")

In [ ]:
print("Interior variables:", list(ds_interior.data_vars))
print("Boundary variables:", list(ds_boundary.data_vars))

In [ ]:
ds_interior["state"].sel(state_feature="t2m").set_index(grid_index=["y", "x"]).unstack("grid_index")

In [ ]:
ds_boundary["forcing"].sel(forcing_feature="2m_temperature")

In [ ]:
plot_interior_vs_boundary(ds_interior, ds_boundary, interior_var="t2m", boundary_var="2m_temperature", plot_time="2020-02-09T12:00:00", zoom_factor=1)